In [1]:
import re
from typing import Dict, List

import chromadb
import pandas as pd
from tqdm import tqdm


def parse_emission_qa_pairs(
    file_path: str, sample_percentage: float = 1.0
) -> List[Dict[str, str]]:
    """
    Parse Q&A pairs from the emissions Q&A file.
    Each Q&A pair becomes a separate document.
    Only processes a sample percentage of the data for faster execution.
    """
    import random

    qa_pairs = []

    with open(file_path, "r", encoding="utf-8") as file:
        content = file.read()

    # Split by double newlines to separate Q&A pairs
    pairs = content.split("\n\n")

    # Calculate how many pairs to sample
    total_pairs = len([p for p in pairs if p.strip()])
    sample_size = max(1, int(total_pairs * sample_percentage))

    print(f"Total Q&A pairs found: {total_pairs}")
    print(f"Sampling {sample_size} pairs ({sample_percentage*100:.1f}%)")

    # Filter out empty pairs first
    valid_pairs = [p for p in pairs if p.strip()]

    # Randomly sample pairs
    sampled_pairs = random.sample(valid_pairs, min(sample_size, len(valid_pairs)))

    # Process pairs with progress bar
    for i, pair in enumerate(tqdm(sampled_pairs, desc="Parsing Emission Q&A pairs")):
        lines = pair.strip().split("\n")

        question = ""
        answer = ""

        for line in lines:
            if line.startswith("Question:"):
                question = line.replace("Question:", "").strip()
            elif line.startswith("Answer:"):
                answer = line.replace("Answer:", "").strip()

        if question and answer:
            qa_pairs.append({"question": question, "answer": answer, "id": f"emission_qa_{i}"})

    return qa_pairs

In [2]:
def prepare_emission_qa_documents(
    file_path: str, sample_percentage: float = 1.0
) -> Dict:
    """
    Convert emission Q&A pairs into ChromaDB-ready documents.
    Each Q&A pair becomes a searchable document.
    """
    qa_pairs = parse_emission_qa_pairs(file_path, sample_percentage)

    documents = []
    metadatas = []
    ids = []

    # Process Q&A pairs with progress bar
    for qa in tqdm(qa_pairs, desc="Preparing emission documents"):
        # Create rich document text for semantic search
        document_text = f"""
        Question: {qa['question']}
        Answer: {qa['answer']}
        
        This Q&A pair provides information about Bitcoin mining emissions tracking, 
        carbon accounting, Scope 1 and Scope 2 activities, and GHG Protocol compliance.
        """.strip()

        # Extract keywords from question and answer for better searchability
        question_words = re.findall(r"\b\w+\b", qa["question"].lower())
        answer_words = re.findall(r"\b\w+\b", qa["answer"].lower())
        all_words = question_words + answer_words

        # Detect emission-related topics
        emission_topics = []
        answer_lower = qa["answer"].lower()
        question_lower = qa["question"].lower()
        combined_text = question_lower + " " + answer_lower
        
        if "scope 1" in combined_text or "scope1" in combined_text:
            emission_topics.append("scope1")
        if "scope 2" in combined_text or "scope2" in combined_text:
            emission_topics.append("scope2")
        if "scope 3" in combined_text or "scope3" in combined_text:
            emission_topics.append("scope3")
        if "electricity" in combined_text or "power" in combined_text:
            emission_topics.append("electricity")
        if "asic" in combined_text or "miner" in combined_text or "mining" in combined_text:
            emission_topics.append("mining_equipment")
        if "cooling" in combined_text or "refrigerant" in combined_text:
            emission_topics.append("cooling")
        if "renewable" in combined_text or "solar" in combined_text or "wind" in combined_text:
            emission_topics.append("renewable_energy")
        if "pue" in combined_text or "efficiency" in combined_text:
            emission_topics.append("efficiency")
        if "carbon" in combined_text or "co2" in combined_text or "ghg" in combined_text:
            emission_topics.append("carbon_accounting")

        # Create metadata for filtering and exact lookups
        metadata = {
            "question": qa["question"],
            "answer": qa["answer"],
            "question_length": len(qa["question"]),
            "answer_length": len(qa["answer"]),
            "keywords": " ".join(set(all_words)),
            "has_question_mark": "?" in qa["question"],
            "topic": "emission_tracking",
            "emission_topics": ",".join(emission_topics) if emission_topics else "general",
            "has_scope1": "scope1" in emission_topics,
            "has_scope2": "scope2" in emission_topics,
            "has_scope3": "scope3" in emission_topics,
        }

        documents.append(document_text)
        metadatas.append(metadata)
        ids.append(qa["id"])

    return {"documents": documents, "metadatas": metadatas, "ids": ids}

In [3]:
def setup_emission_qa_chromadb(
    file_path: str,
    collection_name: str = "emission_qna",
    sample_percentage: float = 1.0,
):
    """
    Create and populate ChromaDB collection with Bitcoin mining emission Q&A data.
    """
    # Initialize ChromaDB
    client = chromadb.PersistentClient("./chroma")

    # Create collection (delete if exists)
    try:
        client.delete_collection(collection_name)
    except:
        pass

    collection = client.create_collection(
        name=collection_name,
        metadata={
            "description": "Bitcoin mining emissions Q&A database with questions and answers about emission tracking, carbon accounting, and GHG Protocol"
        },
    )

    # Prepare documents
    data = prepare_emission_qa_documents(file_path, sample_percentage)

    # Add to ChromaDB with progress bar
    print("Adding documents to ChromaDB...")
    collection.add(
        documents=data["documents"], metadatas=data["metadatas"], ids=data["ids"]
    )

    print(
        f"Added {len(data['documents'])} emission Q&A pairs to ChromaDB collection '{collection_name}'"
    )
    return collection

### Setup the collections

In [4]:
# Create the collection from your emissions Q&A file
collection = setup_emission_qa_chromadb(
    "./data/emissions_questions_output.txt", 
    "emission_qna", 
    sample_percentage=1.0  # Use 100% of data, or 0.05 for 5% sample
)

Total Q&A pairs found: 58
Sampling 58 pairs (100.0%)


Preparing emission documents: 100%|██████████| 58/58 [00:00<00:00, 19583.77it/s]

Adding documents to ChromaDB...


Added 58 emission Q&A pairs to ChromaDB collection 'emission_qna'


### Connect to existing collection

In [5]:
chroma_client = chromadb.PersistentClient("./chroma")
emission_qna = chroma_client.get_collection(name="emission_qna")
print(f"Total Q&A pairs in collection: {emission_qna.count()}")

Total Q&A pairs in collection: 58


### Test queries

In [6]:
print("=== Query 1: ASIC miner electricity tracking ===")
results = emission_qna.query(
    query_texts=["How do I track electricity consumption for ASIC miners?"], 
    n_results=3
)
for i, doc in enumerate(results["documents"][0]):
    print(f"\nResult {i+1}:")
    print(f"Question: {results['metadatas'][0][i]['question']}")
    print(f"Answer: {results['metadatas'][0][i]['answer']}")
    print(f"Topics: {results['metadatas'][0][i]['emission_topics']}")
    print("-" * 80)

=== Query 1: ASIC miner electricity tracking ===

Result 1:
Question: How should I measure electricity consumption for ASIC miners?
Answer: Electricity consumption for ASIC miners should be measured in kilowatt-hours (kWh) using dedicated meters at the miner level or circuit level. Track total kWh consumed, operational hash rate (TH/s or EH/s), and calculate efficiency metrics like J/TH (joules per terahash). This data collection is CRITICAL priority as ASIC electricity represents 95-98% of total emissions for Bitcoin mining operations.
Topics: electricity,mining_equipment,efficiency
--------------------------------------------------------------------------------

Result 2:
Question: How do I track cooling system electricity consumption?
Answer: Cooling system electricity should be tracked separately from ASIC miner consumption, measured in kWh. This includes power for fans, pumps, chillers, and immersion cooling circulation systems. Cooling typically represents 1-3% of total emissions

In [7]:
print("=== Query 2: Scope 2 emissions ===")
results = emission_qna.query(
    query_texts=["What are Scope 2 emissions for Bitcoin mining?"], 
    n_results=3,
    where={"has_scope2": True}
)
for i, doc in enumerate(results["documents"][0]):
    print(f"\nResult {i+1}:")
    print(f"Question: {results['metadatas'][0][i]['question']}")
    print(f"Answer: {results['metadatas'][0][i]['answer']}")
    print("-" * 80)

=== Query 2: Scope 2 emissions ===

Result 1:
Question: What are Scope 2 emissions for cryptocurrency mining facilities?
Answer: Scope 2 emissions are indirect emissions from the consumption of purchased electricity, heat, steam, or cooling. For Bitcoin mining, this is dominated by electricity consumption for ASIC miners, which typically represents 95-98% of total emissions. Additional Scope 2 sources include electricity for cooling systems (1-3%), facility operations, and administrative offices.
--------------------------------------------------------------------------------

Result 2:
Question: What is the typical emission breakdown for a Bitcoin mining facility?
Answer: Typical breakdown: Scope 2 electricity for ASICs (95-98%), Scope 2 cooling electricity (1-3%), Scope 3 upstream electricity (0.5-1.5%), Scope 3 ASIC hardware (1-2%), Scope 2 facility operations (0.1-0.5%), Scope 1 on-site generation if applicable (0.1-2%), all other sources (<0.5% combined). Actual percentages vary b

In [8]:
print("=== Query 3: Cooling systems ===")
results = emission_qna.query(
    query_texts=["cooling system emissions refrigerant"], 
    n_results=3
)
for i, doc in enumerate(results["documents"][0]):
    print(f"\nResult {i+1}:")
    print(f"Question: {results['metadatas'][0][i]['question']}")
    print(f"Answer: {results['metadatas'][0][i]['answer']}")
    print(f"Topics: {results['metadatas'][0][i]['emission_topics']}")
    print("-" * 80)

=== Query 3: Cooling systems ===

Result 1:
Question: What refrigerant leaks do I need to track for my cooling systems?
Answer: Track all refrigerant leaks from HVAC systems and immersion cooling by refrigerant type (R-410A, R-134a, etc.) and quantity in kilograms. This is a Scope 1 fugitive emission with LOW priority (0.01-0.1% of total emissions). Maintain leak detection logs, repair records, and annual refrigerant inventory reconciliations. Different refrigerants have vastly different global warming potentials, so proper classification is essential.
Topics: scope1,cooling
--------------------------------------------------------------------------------

Result 2:
Question: How do I track cooling system electricity consumption?
Answer: Cooling system electricity should be tracked separately from ASIC miner consumption, measured in kWh. This includes power for fans, pumps, chillers, and immersion cooling circulation systems. Cooling typically represents 1-3% of total emissions but is c

In [9]:
print("=== Query 4: Renewable energy ===")
results = emission_qna.query(
    query_texts=["renewable energy solar wind power"], 
    n_results=3
)
for i, doc in enumerate(results["documents"][0]):
    print(f"\nResult {i+1}:")
    print(f"Question: {results['metadatas'][0][i]['question']}")
    print(f"Answer: {results['metadatas'][0][i]['answer']}")
    print("-" * 80)

=== Query 4: Renewable energy ===

Result 1:
Question: How should I document renewable energy installations?
Answer: For owned renewable installations (solar, wind, hydro), document: installed capacity (MW), generation (kWh), capacity factor (%), capital cost, and operational date. Track monthly generation and correlate with mining consumption. This is CRITICAL for Scope 2 emission reduction. Owned renewables provide the highest quality emission reductions with direct temporal matching.
--------------------------------------------------------------------------------

Result 2:
Question: How do I track renewable energy usage at my mining facility?
Answer: Track renewable energy as a percentage of total consumption and by source type (solar, wind, hydro). Document through: (1) direct renewable installations with capacity in MW and generation in kWh, (2) Power Purchase Agreements (PPAs) with contract terms and kWh supplied, (3) Renewable Energy Certificates (RECs) purchased with vintage a

In [10]:
print("=== Query 5: PUE and efficiency metrics ===")
results = emission_qna.query(
    query_texts=["Power Usage Effectiveness PUE efficiency"], 
    n_results=3
)
for i, doc in enumerate(results["documents"][0]):
    print(f"\nResult {i+1}:")
    print(f"Question: {results['metadatas'][0][i]['question']}")
    print(f"Answer: {results['metadatas'][0][i]['answer']}")
    print("-" * 80)

=== Query 5: PUE and efficiency metrics ===

Result 1:
Question: What is Power Usage Effectiveness (PUE) and why is it important?
Answer: Power Usage Effectiveness (PUE) is a critical efficiency metric calculated as total facility energy divided by IT equipment energy. For Bitcoin mining, target PUE should be below 1.2. A PUE of 1.2 means that for every 1 kW used by mining equipment, an additional 0.2 kW is used for cooling and other infrastructure. Lower PUE indicates better efficiency and lower overall emissions per hash rate.
--------------------------------------------------------------------------------

Result 2:
Question: What administrative office electricity data should I collect?
Answer: Collect electricity consumption in kWh for administrative and office spaces separate from mining operations. This typically represents 0.05-0.2% of total emissions with MEDIUM priority. If offices share buildings with mining equipment, use area-based allocation or sub-metering to separate con